In [1]:
import os
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, GlobalAveragePooling2D, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [2]:
# Hyperparameters
IMG_HEIGHT, IMG_WIDTH = 256, 256 
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

class_indices = {
    'Tomato___Bacterial_spot': 0,
    'Tomato___Early_blight': 1,
    'Tomato___Late_blight': 2,
    'Tomato___Leaf_Mold': 3,
    'Tomato___Septoria_leaf_spot': 4,
    'Tomato___Spider_mites Two-spotted_spider_mite': 5,
    'Tomato___Target_Spot': 6,
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus': 7,
    'Tomato___Tomato_mosaic_virus': 8,
    'Tomato___healthy': 9
}
num_classes = len(list(class_indices.keys()))


In [3]:
# Define paths
data_dir = 'data'
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')
test_dir = os.path.join(data_dir, 'test')

# Data Augmentation and Preprocessing
AUTOTUNE = tf.data.experimental.AUTOTUNE

def preprocess(image, label):
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = image / 255.0 
    return image, label

# training dataset
train_data = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    batch_size=BATCH_SIZE,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    label_mode='categorical' 
).map(preprocess).cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)

# validation dataset
val_data = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    batch_size=BATCH_SIZE,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    label_mode='categorical'
).map(preprocess).cache().prefetch(buffer_size=AUTOTUNE)

# test dataset
test_data = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    batch_size=BATCH_SIZE,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    label_mode='categorical'
).map(preprocess).cache().prefetch(buffer_size=AUTOTUNE)


# Implement Parallel and Distributed Training using tf.distribute.Strategy

# Create a MirroredStrategy object to distribute training across available GPUs.
strategy = tf.distribute.MirroredStrategy()
print('Number of devices: {}'.format(strategy.num_replicas_in_sync))

Found 7000 files belonging to 10 classes.
Found 1000 files belonging to 10 classes.
Found 3000 files belonging to 10 classes.
INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)
Number of devices: 1


In [4]:
# Create the model within the strategy scope to ensure distributed training.
with strategy.scope():
    # Define CNN Model
    cnn_model = Sequential([
        tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        Conv2D(16, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(32, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),

        GlobalAveragePooling2D(),

        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])

    cnn_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

cnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 254, 254, 16)   │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 254, 254, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 127, 127, 16)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 125, 125, 32)   │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 125, 125, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 62, 62, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 60, 60, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 60, 60, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 116,202 (453.91 KB)

 Trainable params: 115,722 (452.04 KB)

 Non-trainable params: 480 (1.88 KB)

In [5]:
# Callbacks
callbacks = [
    EarlyStopping(patience=10, verbose=1, restore_best_weights=True),
    ModelCheckpoint('cnn_parallel_model.keras', save_best_only=True, verbose=1)
]

# Train CNN Model using distributed strategy
cnn_history = cnn_model.fit(
    train_data,
    validation_data=val_data,
    epochs=EPOCHS,
    callbacks=callbacks
)



Epoch 1/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 343ms/step - accuracy: 0.5941 - loss: 1.2015
Epoch 1: val_loss improved from inf to 7.42957, saving model to cnn_parallel_model.keras
219/219 ━━━━━━━━━━━━━━━━━━━━ 80s 357ms/step - accuracy: 0.5947 - loss: 1.2000 - val_accuracy: 0.1000 - val_loss: 7.4296
Epoch 2/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step - accuracy: 0.8250 - loss: 0.5097
Epoch 2: val_loss improved from 7.42957 to 3.98255, saving model to cnn_parallel_model.keras
219/219 ━━━━━━━━━━━━━━━━━━━━ 77s 351ms/step - accuracy: 0.8251 - loss: 0.5095 - val_accuracy: 0.2290 - val_loss: 3.9826
Epoch 3/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 333ms/step - accuracy: 0.8781 - loss: 0.3525
Epoch 3: val_loss improved from 3.98255 to 1.42026, saving model to cnn_parallel_model.keras
219/219 ━━━━━━━━━━━━━━━━━━━━ 75s 344ms/step - accuracy: 0.8782 - loss: 0.3523 - val_accuracy: 0.6000 - val_loss: 1.4203
Epoch 4/20
219/219 ━━━━━━━━━━━━━━━━━━━━ 0s 298ms/step - accuracy: 0.9158 - loss: 0.2595
Epoch 4: val

KeyboardInterrupt: 